# Regularization: Ridge, Lasso, and ElasticNet
Accompanies **Chapter 8: Regularization and the Bias-Variance Tradeoff**.

Unregularized OLS estimators suffer from high variance when features are collinear. 
Regularization constrains the parameter space by adding penalty functions to the loss:
- **Ridge ($L_2$):** $\mathcal{L}_{\text{Ridge}}(\mathbf{w}) = \|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha \|\mathbf{w}\|_2^2$. Shifts eigenvalues of $\mathbf{X}^T \mathbf{X}$ by $+\alpha$, shrinking weights smoothly toward zero without selecting features.
- **Lasso ($L_1$):** $\mathcal{L}_{\text{Lasso}}(\mathbf{w}) = \frac{1}{2n}\|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha \|\mathbf{w}\|_1$. The non-differentiable sharp corners of the $L_1$ ball force weak coefficients to hit zero exactly, performing automated feature selection.

In this notebook, we trace the coefficient paths of both regularizers across 5 orders of magnitude.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression

# 150 observations, 20 features, only 5 genuinely informative
X, y, true_coef = make_regression(n_samples=150, n_features=20, n_informative=5, noise=12.0, coef=True, random_state=42)

# Regularizers assume features are on identical scales
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Tracing the Regularization Paths
We sweep penalty parameter $\alpha$ from $10^{-2}$ to $10^{3}$. 
Notice the structural difference:
- In Ridge, coefficients approach zero asymptotically but remain strictly non-zero.
- In Lasso, coefficients drop to zero abruptly at distinct threshold values of $\alpha$.

In [ ]:
alphas = np.logspace(-2, 3, 100)
ridge_coefs = []
lasso_coefs = []

for a in alphas:
    r = Ridge(alpha=a).fit(X_scaled, y)
    l = Lasso(alpha=a, max_iter=3000).fit(X_scaled, y)
    ridge_coefs.append(r.coef_)
    lasso_coefs.append(l.coef_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8))

ax1.plot(alphas, ridge_coefs)
ax1.set_xscale('log')
ax1.set_xlabel('Regularization Strength $\\alpha$', fontsize=11)
ax1.set_ylabel('Weight Value ($w_j$)', fontsize=11)
ax1.set_title('Ridge ($L_2$) Path: Smooth Weight Shrinkage', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(alphas, lasso_coefs)
ax2.set_xscale('log')
ax2.set_xlabel('Regularization Strength $\\alpha$', fontsize=11)
ax2.set_ylabel('Weight Value ($w_j$)', fontsize=11)
ax2.set_title('Lasso ($L_1$) Path: Coordinate Sparsity', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Cross-Validated Tuning of $\alpha$
We use 5-fold cross-validation (`LassoCV`) to determine the minimum mean-squared-error penalty.

In [ ]:
lasso_cv = LassoCV(cv=5, random_state=42).fit(X_scaled, y)
optimal_alpha = lasso_cv.alpha_
selected_count = np.sum(lasso_cv.coef_ != 0)

print(f"Optimal Alpha chosen by CV: {optimal_alpha:.4f}")
print(f"Non-zero features selected:  {selected_count} out of 20")